# Challenge 04 -- Deploy a Hosted Weather Agent on Foundry (SOLUTION)

This is the **completed solution** notebook. All blanks are filled in.

Refer to `Student/Challenge-04.md` for the full challenge description and learning context.

---
## Setup -- Install Dependencies and Authenticate

Ensure your `.env` file (created in Challenge 00) is in the working directory, then run these cells.

In [ ]:
%pip install azure-ai-projects azure-identity python-dotenv pyyaml --quiet

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

PROJECT_ENDPOINT = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
MODEL_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
RESOURCE_GROUP = os.getenv("AZURE_RESOURCE_GROUP")

assert PROJECT_ENDPOINT, "AZURE_AI_FOUNDRY_ENDPOINT not found in .env"
assert MODEL_DEPLOYMENT_NAME, "AZURE_OPENAI_DEPLOYMENT_NAME not found in .env"
assert RESOURCE_GROUP, "AZURE_RESOURCE_GROUP not found in .env"

print(f"Endpoint:       {PROJECT_ENDPOINT}")
print(f"Model:          {MODEL_DEPLOYMENT_NAME}")
print(f"Resource Group: {RESOURCE_GROUP}")

In [ ]:
import subprocess, json

# Create an ACR (or reuse if it already exists in this resource group)
rg_result = subprocess.run(
    f"az acr list --resource-group {RESOURCE_GROUP} --query [0].name -o tsv",
    shell=True, capture_output=True, text=True,
)
existing_acr = rg_result.stdout.strip()

if existing_acr:
    ACR_NAME = existing_acr
    print(f"Using existing ACR: {ACR_NAME}")
else:
    import random, string
    ACR_NAME = "acr" + "".join(random.choices(string.ascii_lowercase + string.digits, k=10))
    result = subprocess.run(
        f"az acr create --name {ACR_NAME} --resource-group {RESOURCE_GROUP} --sku Basic --admin-enabled true",
        shell=True, capture_output=True, text=True,
    )
    if result.returncode == 0:
        print(f"Created ACR: {ACR_NAME}")
    else:
        print(f"Failed to create ACR: {result.stderr}")

print(f"ACR: {ACR_NAME}")

In [ ]:
from azure.identity import AzureCliCredential, InteractiveBrowserCredential, ChainedTokenCredential

TENANT_ID = os.getenv("AZURE_TENANT_ID")

credential = ChainedTokenCredential(
    AzureCliCredential(tenant_id=TENANT_ID) if TENANT_ID else AzureCliCredential(),
    InteractiveBrowserCredential(tenant_id=TENANT_ID) if TENANT_ID else InteractiveBrowserCredential(),
)

credential.get_token("https://management.azure.com/.default")
print("Connected to Azure")

---
## Part 1: Define the Weather Tool

Write a tool function that the agent can call to look up weather data. Use `Annotated`
with a `Field` description so the agent knows what the parameter means.

In [ ]:
from typing import Annotated
from pydantic import Field

# Simulated weather data -- in production this would call a real weather API
WEATHER_DATA = {
    "seattle": "55F, cloudy with light rain",
    "new york": "72F, sunny with a few clouds",
    "london": "60F, overcast with drizzle",
    "tokyo": "80F, warm and humid",
    "paris": "63F, clear skies",
}

def get_weather(
    location: Annotated[str, Field(description="The city or location to get weather for")]
) -> str:
    '''Get the current weather for a given location.'''
    result = WEATHER_DATA.get(location.lower(), f"65F, partly cloudy in {location}")
    return f"The weather in {location} is: {result}"

# Test it right here
print(get_weather("Seattle"))
print(get_weather("Tokyo"))
print(get_weather("Narnia"))

---
## Part 2: Write Agent Instructions

Write the instructions that define your agent's persona and behavior. These go into
the hosted agent code and control how it responds to users.

In [ ]:
AGENT_INSTRUCTIONS = (
    "You are a helpful weather assistant. When a user asks about the weather, "
    "use the get_weather tool to look up the current conditions. Always mention "
    "the location in your response and provide a concise, friendly summary. "
    "If the user asks about a location not in your data, provide the default "
    "conditions and let them know the data is simulated."
)

print("Instructions length:", len(AGENT_INSTRUCTIONS), "chars")
print()
print(AGENT_INSTRUCTIONS)

---
## Part 3: Assemble the Agent Code

This cell combines your tool function and instructions into `main.py` -- the hosted
agent application. Run as-is.

In [ ]:
import inspect, textwrap

# Get the source of the tool function and weather data you defined above
tool_source = inspect.getsource(get_weather)
data_source = "WEATHER_DATA = " + repr(WEATHER_DATA)

# Write the complete main.py
os.makedirs("weather-agent", exist_ok=True)

with open("weather-agent/main.py", "w") as f:
    f.write('''import os
from typing import Annotated

from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from agent_framework_foundry_hosting import ResponsesHostServer
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from pydantic import Field

load_dotenv()


# --- Weather data and tool ---

''' + data_source + '''


@tool(approval_mode="never_require")
''' + tool_source + '''


# --- Create the Foundry chat client ---

client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=DefaultAzureCredential(),
)


# --- Create the agent ---

agent = Agent(
    client=client,
    instructions=''' + repr(AGENT_INSTRUCTIONS) + ''',
    tools=[get_weather],
    default_options={"store": False},
)


# --- Start the hosted server ---

server = ResponsesHostServer(agent)
server.run()
''')

# Verify
import py_compile
py_compile.compile("weather-agent/main.py", doraise=True)
print("[OK] weather-agent/main.py written and syntax-valid")

# Show the result
with open("weather-agent/main.py") as f:
    for i, line in enumerate(f, 1):
        print(f"{i:3d}  {line}", end="")

---
## Part 4: Configure the Agent Definition

Write `agent.yaml`, the Dockerfile, and supporting files. Run as-is.

In [ ]:
import yaml

agent_config = {
    "kind": "hosted",
    "name": "weather-agent",
    "protocols": [
        {"protocol": "responses", "version": "1.0.0"}
    ],
    "resources": {
        "cpu": "0.5",
        "memory": "1Gi",
    },
    "environment_variables": [
        {"name": "AZURE_AI_MODEL_DEPLOYMENT_NAME", "value": MODEL_DEPLOYMENT_NAME}
    ],
}

with open("weather-agent/agent.yaml", "w") as f:
    yaml.dump(agent_config, f, default_flow_style=False, sort_keys=False)

print("[OK] weather-agent/agent.yaml written")
print()
with open("weather-agent/agent.yaml") as f:
    print(f.read())

In [ ]:
# Write Dockerfile and supporting files
with open("weather-agent/Dockerfile", "w") as f:
    f.write("""FROM python:3.12-slim

WORKDIR /app

COPY . user_agent/
WORKDIR /app/user_agent

RUN if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

EXPOSE 8088

CMD ["python", "main.py"]
""")

with open("weather-agent/requirements.txt", "w") as f:
    f.write("""agent-framework>=1.2.2
agent-framework-foundry-hosting
azure-identity
python-dotenv
pydantic
""")

with open("weather-agent/.dockerignore", "w") as f:
    f.write(""".venv
__pycache__
*.pyc
.env
""")

# Show project structure
print("[OK] All agent files written")
print()
print("weather-agent/")
for fname in sorted(os.listdir("weather-agent")):
    if not fname.startswith("__"):
        size = os.path.getsize(f"weather-agent/{fname}")
        print(f"  {fname} ({size} bytes)")

---
## Part 5: Build the Container Image

Build the Docker image remotely on Azure Container Registry. Takes about 2 minutes. Run as-is.

In [ ]:
import json as _json
import subprocess

print("Building container image on ACR (this takes about 2 minutes)...")

result = subprocess.run(
    f"az acr build --registry {ACR_NAME} --image weather-agent:v1 "
    f"--platform linux/amd64 weather-agent/ --no-logs",
    shell=True, capture_output=True, text=True,
)

if result.returncode == 0:
    build_info = _json.loads(result.stdout)
    status = build_info.get("status", "unknown")
    images = build_info.get("outputImages", [])
    print(f"Build status: {status}")
    if images:
        print(f"Image: {images[0]['registry']}/{images[0]['repository']}:{images[0]['tag']}")
    if status == "Succeeded":
        print("[OK] Container image built successfully")
    else:
        print("[WARN] Build finished but status is: " + status)
else:
    print("[FAIL] Build failed:")
    print(result.stderr[:500])

---
## Part 6: Deploy the Agent to Foundry

Use the Python SDK to deploy your container as a hosted agent on Foundry.

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import HostedAgentDefinition, ProtocolVersionRecord, AgentProtocol

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
    allow_preview=True,
)

# Check if any version is already active (avoids creating duplicates on re-run)
active_version = None
try:
    for v in project.agents.list_versions(agent_name="weather-agent"):
        if v["status"] == "active":
            active_version = v
            break
except Exception:
    pass

if active_version:
    print(f"Agent weather-agent v{active_version.version} is already active -- skipping creation")
    agent_version = active_version
else:
    image_url = f"{ACR_NAME}.azurecr.io/weather-agent:v1"

    agent_version = project.agents.create_version(
        agent_name="weather-agent",
        definition=HostedAgentDefinition(
            container_protocol_versions=[
                ProtocolVersionRecord(protocol=AgentProtocol.RESPONSES, version="1.0.0")
            ],
            cpu="0.5",
            memory="1Gi",
            image=image_url,
            environment_variables={
                "AZURE_AI_MODEL_DEPLOYMENT_NAME": MODEL_DEPLOYMENT_NAME,
            },
        ),
    )
    print(f"Agent created: {agent_version.name}, version: {agent_version.version}")

In [ ]:
import time

version_num = str(agent_version.version)

while True:
    info = project.agents.get_version(agent_name="weather-agent", agent_version=version_num)
    status = info["status"]
    print(f"Status: {status}")
    if status == "active":
        print("[OK] Agent is deployed and ready")
        break
    elif status == "failed":
        print("[FAIL] Provisioning failed")
        break
    time.sleep(10)

---
## Part 7: Invoke the Deployed Agent

Send requests to the hosted agent running on Foundry.

In [ ]:
openai_client = project.get_openai_client(agent_name="weather-agent")

response = openai_client.responses.create(
    input="What is the weather like in Paris right now?",
)

print("Deployed agent says:")
print(response.output_text)

In [ ]:
response2 = openai_client.responses.create(
    input="How about Tokyo? Is it warmer than Paris?",
)

print("Deployed agent says:")
print(response2.output_text)

---
## Verify in the Foundry Portal

Open the [Microsoft Foundry portal](https://ai.azure.com) and confirm your Weather Agent
appears under the **Agents** section with **Active** status.

In [ ]:
print("=" * 55)
print("CHALLENGE 04 -- COMPLETION CHECKLIST")
print("=" * 55)
print()
print("[x]  Defined a weather tool with typed parameters")
print("[x]  Wrote agent instructions controlling persona and behavior")
print("[x]  Assembled main.py from tool + instructions + hosting stack")
print("[x]  Configured agent.yaml (protocol, resources, env vars)")
print("[x]  Built container image on ACR")
print("[x]  Deployed agent via Python SDK")
print("[x]  Invoked deployed agent -- got weather responses")
print()
print("Show your deployed agent and responses to your coach!")

---
## Cleanup

Delete the agent when finished (uncomment to run):
```python
# project.agents.delete_version(agent_name="weather-agent", agent_version=str(agent_version.version))
```